# EDA Crossover

# Setup

## Imports

In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import ticker as mtick
from tqdm.auto import tqdm

from thesis_project import (
    config,
)

## Paths

In [ ]:
TICKER = "fbtp"
# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # NOTE: Create a purpose specific directory if needed
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

In [ ]:
logger = logging.getLogger(__name__)

## Utilities

# Main

## Common

In [ ]:
# Load data
TARGET = "basis"
FILTER = True

if TARGET == "basis":
    filename = "gross_basis_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
if TARGET == "irr":
    filename = "irr_freq_1s_from_2022_08_01_to_2024_12_30.parquet"

target_df = pd.read_parquet(config.INT_DIR / TICKER / "data-microstructure" / TARGET / filename)
target_df = target_df.between_time("09:00:00", "17:00:00")
target_df = target_df.astype("float32")

assert isinstance(target_df.index, pd.DatetimeIndex)
if not target_df.index.is_monotonic_increasing:
    target_df = target_df.sort_index()

if FILTER:
    import thesis_project.importance_mi.data_pipeline as data_pipeline

    filter_dates = data_pipeline.filter_dates

    filters = {
        "fut_last_trading_days": [5, 0],
    }
    target_df = filter_dates(target_df, TICKER, filters)

if TARGET == "basis":
    target_s = target_df["Basis"]
if TARGET == "irr":
    target_s = target_df["IRR"]

## Target Vs Rolling Mean

### Utilities

In [ ]:
from typing import cast


def compute_crossover_time(
    series: pd.Series,
    window: int,
    eps: float | None = None,
):
    # Validate inputs
    if not isinstance(series.index, pd.DatetimeIndex):
        raise ValueError("Series index must be a DatetimeIndex")

    if window <= 0:
        raise ValueError("Window must be a positive integer")

    if eps is not None and eps < 0:
        raise ValueError("Epsilon must be non-negative")

    # Clean data
    if series.index.has_duplicates:
        logger.debug(
            f"Series has {series.index.duplicated().sum()} duplicate timestamps. "
            "Keeping the first occurrence."
        )
        series = series[~series.index.duplicated(keep="first")]

    if not series.index.is_monotonic_increasing:
        logger.debug("Series index is not ordered. Sorting by index.")
        series = series.sort_index()

    timestamp_to_pos = {timestamp: pos for pos, timestamp in enumerate(series.index)}

    # Group by day
    assert isinstance(series.index, pd.DatetimeIndex)
    normalized_dates = series.index.normalize()
    grouped_days = series.groupby(normalized_dates)
    day_list = cast(list[pd.Timestamp], list(grouped_days.groups.keys()))

    daily_groups = {date: group for date, group in grouped_days}

    # Allocate output
    results = {
        "Delta": np.full(len(series), np.nan, dtype=np.float64),
        "Time to Crossover": np.full(len(series), np.nan, dtype=np.float32),
    }

    means = {date: np.nan for date, _ in grouped_days}

    n_obs = len(series)
    n_days = len(day_list)
    logger.info("Starting Crossover Analysis:")
    logger.info(f"  Target: {series.name}")
    logger.info(f"  Series length: {n_days} days ({n_obs} timestamps)")
    logger.info(f"  Lookback window: {window} days")
    logger.info(f"  Epsilon: {eps}")

    if not logger.isEnabledFor(logging.DEBUG):
        grouped_days = tqdm(
            grouped_days,
            desc="Processing days",
            unit="day",
            colour="black",
        )

    logger.debug(" ")

    for day_number, (current_day, current_day_series) in enumerate(grouped_days):
        # Buffer
        if day_number < window:
            logger.debug(f"Skipping {current_day.date()}: " "not enough lookback days.")
            continue

        # Select lookback sample
        lookback_days = day_list[day_number - window : day_number]
        lookback_series = pd.concat([daily_groups[d] for d in lookback_days])

        # Skip day if lookback data are not valid
        if lookback_series.empty:
            logger.debug(f"Skipping {current_day.date()}: " "lookback sample is empty.")
            continue

        lookback_mean = lookback_series.mean()

        # Skip day if mean is not valid
        if pd.isna(lookback_mean):
            logger.debug(f"Skipping {current_day.date()}: " "lookback mean is NaN.")
            continue
        means[current_day] = lookback_mean

        logger.debug(f"Processing {current_day.date()}")

        day_index = current_day_series.index
        day_values = current_day_series.to_numpy()

        # Process day
        for intra_day_pos, (t, X_t) in enumerate(zip(day_index, day_values)):
            t_idx = timestamp_to_pos[t]

            # Skip timestamp if data not valid
            if pd.isna(X_t):
                continue

            delta_t = X_t - lookback_mean
            results["Delta"][t_idx] = delta_t

            # Skip timestamp if end of day
            if intra_day_pos == len(day_values) - 1:
                continue

            # Skip timestamp if delta is zero, exact behavior to be defined
            if np.isclose(delta_t, 0):
                logger.debug(f"  Skipping {t}:" "Behavior not defined for delta_t = 0")
                continue

            sign_t = np.sign(delta_t)

            # Search first crossover after t
            found_valid_future = False
            found_crossover = False
            for future_pos in range(intra_day_pos + 1, len(day_values)):
                X_s = day_values[future_pos]

                if pd.isna(X_s):
                    continue

                found_valid_future = True

                delta_s = X_s - lookback_mean
                sign_s = np.sign(delta_s)

                if sign_s != sign_t:
                    s = day_index[future_pos]
                    results["Time to Crossover"][t_idx] = (s - t).total_seconds()
                    found_crossover = True
                    break

            # Assign inf if there are valid data for the analysis but no crossover spotted
            if found_valid_future and not found_crossover:
                results["Time to Crossover"][t_idx] = np.inf

    # Summary
    summary = {
        "Delta": {
            "Finite": np.isfinite(results["Delta"]).sum(),
            "Finite (%)": np.isfinite(results["Delta"]).mean() * 100,
            "NaN": np.isnan(results["Delta"]).sum(),
            "NaN (%)": np.isnan(results["Delta"]).mean() * 100,
        },
        "Time to Crossover": {
            "Finite": np.isfinite(results["Time to Crossover"]).sum(),
            "Finite (%)": np.isfinite(results["Time to Crossover"]).mean() * 100,
            "NaN": np.isnan(results["Time to Crossover"]).sum(),
            "NaN (%)": np.isnan(results["Time to Crossover"]).mean() * 100,
            "Inf": np.isinf(results["Time to Crossover"]).sum(),
            "Inf (%)": np.isinf(results["Time to Crossover"]).mean() * 100,
        },
    }

    logger.info(" ")
    logger.info("Finished crossover analysis")
    logger.info("Results summary:")

    for variable_name, stats in summary.items():
        logger.info(f"  {variable_name}:")
        logger.info(f"    Finite: {stats['Finite']} " f"({stats['Finite (%)']:.2f}%)")
        logger.info(f"    NaN: {stats['NaN']} " f"({stats['NaN (%)']:.2f}%)")

        if variable_name == "Time to Crossover":
            logger.info(f"    Inf: {stats['Inf']} " f"({stats['Inf (%)']:.2f}%)")

    return pd.DataFrame(results, index=series.index), pd.Series(means, index=means.keys())

In [ ]:
def filter_crossovers(previous, current, first):
    if first:
        return not (np.isnan(current) or np.isinf(current))

    if np.isnan(current) or np.isinf(current):
        return False

    if np.isnan(previous) or np.isinf(previous):
        return True

    if np.isclose(current, previous - 1):
        return False

    return True

### Run Analysis

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-5s | %(message)s",
    force=True,
)

results = compute_crossover_time(target_s, 1)

path = config.INT_DIR / TICKER.lower() / "data-microstructure" / TARGET.lower()
filename = f"{TARGET.lower()}_crossover_analysis"
# results.to_parquet(path / filename)

### Load results

In [ ]:
# Load data
TARGET = "irr"
FILTER = True

map_target = {"basis": "Basis", "irr": "IRR"}

if TARGET == "basis":
    filename = "gross_basis_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
if TARGET == "irr":
    filename = "irr_freq_1s_from_2022_08_01_to_2024_12_30.parquet"

target_df = pd.read_parquet(config.INT_DIR / TICKER / "data-microstructure" / TARGET / filename)
target_df = target_df.between_time("09:00:00", "17:00:00")
target_df = target_df.astype("float32")

assert isinstance(target_df.index, pd.DatetimeIndex)
if not target_df.index.is_monotonic_increasing:
    target_df = target_df.sort_index()

if FILTER:
    import thesis_project.importance_mi.data_pipeline as data_pipeline

    filter_dates = data_pipeline.filter_dates

    filters = {
        "fut_last_trading_days": [5, 0],
    }
    target_df = filter_dates(target_df, TICKER, filters)

target_s = target_df[map_target[TARGET]]

In [ ]:
path = config.INT_DIR / TICKER.lower() / "data-microstructure" / TARGET.lower()
filename = f"{TARGET.lower()}_crossover_analysis"
results = pd.read_parquet(path / filename)

THRESHOLDS = {"basis": [0, 0.05, 0.1, 0.15, 0.2], "irr": [0]}

thresholds = THRESHOLDS[TARGET]

ttc = results["Time to Crossover"]

# Replace inf with NaN
finite = ttc.replace([np.inf, -np.inf], np.nan)

# Previous finite value skipping nans
prev_finite = finite.ffill().shift(1)
is_continuation = finite == (prev_finite - 1)

# First occurrence of a crossover event
results["First Crossover"] = np.isfinite(ttc) & ~is_continuation

### Data Integrity

In [ ]:
data = results.copy()

summary = {
    "Delta": {
        "Finite": np.isfinite(data["Delta"]).sum(),
        "Finite (%)": np.isfinite(data["Delta"]).mean() * 100,
        "Infinite": None,
        "Infinite (%)": None,
        "NaN": np.isnan(data["Delta"]).sum(),
        "NaN (%)": np.isnan(data["Delta"]).mean() * 100,
    },
    "Time to Crossover": {
        "Finite": np.isfinite(data["Time to Crossover"]).sum(),
        "Finite (%)": np.isfinite(data["Time to Crossover"]).mean() * 100,
        "Infinite": np.isinf(data["Time to Crossover"]).sum(),
        "Infinite (%)": np.isinf(data["Time to Crossover"]).mean() * 100,
        "NaN": np.isnan(data["Time to Crossover"]).sum(),
        "NaN (%)": np.isnan(data["Time to Crossover"]).mean() * 100,
    },
}

summary = pd.DataFrame(summary).T
display(summary)

mask = data["First Crossover"]
data = data[mask]

summary = {
    "Delta": {
        "Finite": np.isfinite(data["Delta"]).sum(),
        "Finite (%)": np.isfinite(data["Delta"]).mean() * 100,
        "Infinite": None,
        "Infinite (%)": None,
        "NaN": np.isnan(data["Delta"]).sum(),
        "NaN (%)": np.isnan(data["Delta"]).mean() * 100,
    },
    "Time to Crossover": {
        "Finite": np.isfinite(data["Time to Crossover"]).sum(),
        "Finite (%)": np.isfinite(data["Time to Crossover"]).mean() * 100,
        "Infinite": np.isinf(data["Time to Crossover"]).sum(),
        "Infinite (%)": np.isinf(data["Time to Crossover"]).mean() * 100,
        "NaN": np.isnan(data["Time to Crossover"]).sum(),
        "NaN (%)": np.isnan(data["Time to Crossover"]).mean() * 100,
    },
}

summary = pd.DataFrame(summary).T
display(summary)

### Summary

In [ ]:
columns = ["Delta", "Time to Crossover"]

data = results.copy()

summary = {}
for col in columns:
    temp = data[col]
    mask = np.isfinite(temp)
    temp = temp[mask]
    summary[col] = temp.describe()
    summary[col]["skewness"] = temp.skew()
    summary[col]["kurtosis"] = temp.kurtosis()

summary = pd.DataFrame(summary).T
display(summary)
summary.to_csv(config.TAB_DIR / TICKER / "misc" / f"{TARGET}_crossover_analysis_summary.csv")

mask = data["First Crossover"]
data = data[mask]

summary = {}
for col in columns:
    temp = data[col]
    mask = np.isfinite(temp)
    temp = temp[mask]
    summary[col] = temp.describe()
    summary[col]["skewness"] = temp.skew()
    summary[col]["kurtosis"] = temp.kurtosis()

summary = pd.DataFrame(summary).T
display(summary)
summary.to_csv(
    config.TAB_DIR / TICKER / "misc" / f"{TARGET}_crossover_analysis_summary_filtered.csv"
)

### Delta distribution

In [ ]:
data = results.copy()
mask = data["First Crossover"]
data = data[mask]

col = "Delta"
# threshold = 0.1
# mask = (data["Delta"] >= +threshold) | (data["Delta"] <= -threshold)
# data = data[mask]

data = data[col]
mask = np.isfinite(data)
data = data[mask]

display(pd.DataFrame(data.describe()).T)

lower = 0.000
upper = 1
left = data.quantile(lower)
right = data.quantile(upper)
mask = data.between(left, right)
data = data[mask]

fig, ax = plt.subplots(figsize=(12, 6))

weights = np.ones_like(data) / len(data)
ax.hist(
    data,
    bins=1000,
    weights=weights,
)

ax.set_xlabel(f"{col}")

ax.set_ylabel("Frequency (log)")
ax.set_yscale("log")
if not ax.get_yscale() == "log":
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

title = f"{TARGET.capitalize()} {col} Distribution"
ax.set_title(title, fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(
    config.FIG_DIR / TICKER / "misc" / f"{TARGET}_crossover_analysis_delta_distribution", dpi=300
)

plt.show()

### Time to Crossover distribution

In [ ]:
rows = []

for threshold in thresholds:
    data = results.copy()

    mask = data["First Crossover"]
    data = data[mask]

    mask = (data["Delta"] >= threshold) | (data["Delta"] <= -threshold)
    data = data[mask]

    col = "Time to Crossover"
    data = data[col]

    mask = np.isfinite(data)
    data = data[mask]

    lower = 0.0
    upper = 1.0
    left = data.quantile(lower)
    right = data.quantile(upper)

    mask = data.between(left, right)
    data = data[mask]

    fig, ax = plt.subplots(figsize=(12, 6))

    bins = data.nunique()
    weights = np.ones_like(data) / len(data)

    ax.hist(
        data,
        bins=bins,
        weights=weights,
    )

    ax.set_xlabel("Seconds to Crossover")

    ax.set_ylabel("Frequency (log)")
    ax.set_yscale("log")

    if ax.get_yscale() != "log":
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

    title = f"{TARGET.capitalize()} Time to Crossover Distribution"
    title += f"\nFiltered for $|\\Delta_t| \\geq$ {threshold}" if threshold != 0 else ""
    ax.set_title(title, fontsize=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(
        config.FIG_DIR
        / TICKER
        / "misc"
        / f"{TARGET}_crossover_analysis_ttc_distribution_threshold_{threshold}.png",
        dpi=300,
    )

    plt.show()

    if threshold != 0:
        summary = data.describe().to_dict()
        summary["threshold"] = threshold
        rows.append(summary)

summaries = pd.DataFrame(rows).set_index("threshold").sort_index()
summaries.to_csv(
    config.TAB_DIR / TICKER / "misc" / f"{TARGET}_crossover_analysis_ttc_summary_thresholds.csv"
)

display(summaries)

In [ ]:
map_sign = {+1: "Positive", -1: "Negative"}

rows = []

for threshold in thresholds:
    data = results.copy()

    mask = data["First Crossover"]
    data = data[mask]

    mask = (data["Delta"] >= threshold) | (data["Delta"] <= -threshold)
    data = data[mask]

    mask = np.isfinite(data).all(axis=1)
    data = data[mask]

    col = "Time to Crossover"
    temp = data.copy()

    all_times = temp[col]
    xmin = int(all_times.min())
    xmax = int(all_times.max())
    bins = np.arange(xmin - 0.5, xmax + 1.5, 1)

    fig, ax = plt.subplots(figsize=(12, 6))

    for sign in [-1, +1]:
        mask = np.sign(temp["Delta"]) == sign
        data = temp.loc[mask, col]

        if threshold != 0:
            summary = data.describe().to_dict()
            summary["threshold"] = threshold
            summary["sign"] = map_sign[sign]
            rows.append(summary)

        lower = 0
        upper = 1
        left = data.quantile(lower)
        right = data.quantile(upper)

        data_plot = data[data.between(left, right)]

        # bins = data_plot.nunique()
        weights = np.ones_like(data_plot) / len(data_plot)

        label = f"Delta {map_sign[sign].lower()}"

        ax.hist(
            data_plot,
            bins=bins,
            weights=weights,
            alpha=0.7,
            label=label,
        )

    ax.set_xlabel("Seconds to Crossover")
    ax.set_ylabel("Frequency (log)")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_yscale("log")

    title = f"{TARGET.capitalize()} Time to Crossover Distribution"
    title += f"\nFiltered for $|\\Delta_t| \\geq$ {threshold}" if threshold != 0 else ""
    ax.set_title(title, fontsize=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(loc="upper right", fontsize=10)

    fig.tight_layout()
    fig.savefig(
        config.FIG_DIR
        / TICKER
        / "misc"
        / f"{TARGET}_crossover_analysis_ttc_distribution_threshold_{threshold}_signs.png",
        dpi=300,
    )

    plt.show()

# Final summary dataframe
summaries = pd.DataFrame(rows).set_index(["threshold", "sign"]).sort_index()
summaries.to_csv(
    config.TAB_DIR
    / TICKER
    / "misc"
    / f"{TARGET}_crossover_analysis_ttc_summary_thresholds_signs.csv"
)

display(summaries)

### Conditioned Time to Crossover

In [ ]:
n_quantiles = 20

for threshold in thresholds:
    data = results.copy()
    mask = data["First Crossover"]
    data = data[mask]

    mask = (data["Delta"] >= threshold) | (data["Delta"] <= -threshold)
    data = data[mask]

    mask = np.isfinite(data).all(axis=1)
    data = data[mask]

    bins = pd.qcut(data["Delta"], q=n_quantiles)
    x_labels = [f"({bin.left:.3f}, {bin.right:.3f}]" for bin in bins.cat.categories]

    data["bin"] = pd.qcut(data["Delta"], q=n_quantiles)

    fig, ax = plt.subplots(figsize=(20, 8))

    data.boxplot(
        column="Time to Crossover",
        by="bin",
        ax=ax,
        showfliers=False,
        color="black",
    )
    fig.suptitle("")

    means = data.groupby("bin", observed=True)["Time to Crossover"].mean()

    ax.plot(range(1, len(means) + 1), means, marker="x", linestyle="-", label="Mean", color="red")

    ax.set_xlabel("Delta quantiles")
    ax.set_xticklabels(x_labels)

    ax.set_ylabel("Time to Crossover")

    title = (
        f"{TARGET.capitalize()} Time to Crossover distribution" f"\nConditioned by Delta quantiles"
    )
    title += f"\nFiltered for $|\\Delta_t| \\geq$ {threshold}" if threshold != 0 else ""
    ax.set_title(title, fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x")
    ax.legend(loc="upper right", fontsize=12)

    fig.tight_layout()
    fig.savefig(
        config.FIG_DIR
        / TICKER
        / "misc"
        / f"{TARGET}_crossover_analysis_ttc_boxplot_threshold_{threshold}.png",
        dpi=300,
    )

    plt.show()

In [ ]:
n_quantiles = 10

data = results.copy()
data = data.join(target_df, how="inner")
mask = data["First Crossover"]
data = data[mask]

threshold = 0.1
mask = (data["Delta"] >= threshold) | (data["Delta"] <= -threshold)
data = data[mask]

mask = np.isfinite(data).all(axis=1)
data = data[mask]

bin_col = map_target[TARGET]
data["bin"] = pd.qcut(data[bin_col], q=n_quantiles)

fig, ax = plt.subplots(figsize=(12, 6))

data.boxplot(
    column="Time to Crossover",
    by="bin",
    ax=ax,
    showfliers=True,
)
fig.suptitle("")

ax.set_xlabel(f"{TARGET.capitalize()} quantiles")
ax.set_xticklabels([str(i) for i in range(1, n_quantiles + 1)])

ax.set_ylabel("Time to Crossover")

title = (
    f"{TARGET.capitalize()} Time to Crossover distribution"
    f"\nConditioned by {TARGET.capitalize()} quantiles"
    f"\nFiltered for $|\\Delta_t| \\geq$ {threshold}"
)
ax.set_title(title, fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x")

fig.tight_layout()
plt.show()